<a href="https://colab.research.google.com/github/temesgenaddise/cosc-650-applied-llm-systems/blob/main/Week-3/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week-3: Prompts as Engineering Artifacts Assignmnet: Security Code-Review Triage

# Libraries and Packages

In [1]:
import os, json, pathlib
def gemini_chat(messages, model='gemini-2.5-flash-lite', **kw):
    """Gemini via the OpenAI-compatible endpoint. Returns text, or None if no key (API-BLOCKED)."""
    key = os.environ.get('GEMINI_API_KEY')
    if not key:
        return None
    from openai import OpenAI
    client = OpenAI(api_key=key, base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

LIVE = os.environ.get('GEMINI_API_KEY') is not None
print('live model calls:', LIVE, '(fixtures used when False)')

live model calls: False (fixtures used when False)


In [2]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
def exact_match(a, b):
    return float(str(a).strip().lower() == str(b).strip().lower())
def semantic_sim(a, b):
    e = emb.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return round(float(util.cos_sim(e[0], e[1])), 3)
print('metrics ready (exact-match + semantic)')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

metrics ready (exact-match + semantic)


# Part 1: Build the structured prompts

The three cells below:

. mounts Google Drive in Colab so the notebook can access the external, versioned prompt files stored in the user's Drive.

. checks whether the prompt_versions directory exists in Google Drive and lists the files it contains.

. confirms that the expected prompt versions are available before loading them.

. Defines the paths for v1.txt and v2.txt, verifies that neither file is missing, reads both prompts into memory, and reports their locations and character counts.

In [3]:
# Load the two complete prompt versions from external versioned files.

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from pathlib import Path

drive_prompt_directory = Path('/content/drive/MyDrive/prompt_versions')

if drive_prompt_directory.exists() and drive_prompt_directory.is_dir():
    print(f"Contents of '{drive_prompt_directory}':")
    for item in drive_prompt_directory.iterdir():
        print(f"- {item.name}")
else:
    print(f"The directory '{drive_prompt_directory}' does NOT exist in your Google Drive, so there are no contents to list.")

Contents of '/content/drive/MyDrive/prompt_versions':
- v2.txt
- v1.txt
- live_response_cache.json


In [5]:
from pathlib import Path

PROMPT_DIRECTORY = Path('/content/drive/MyDrive/prompt_versions')
PROMPT_V1_PATH = PROMPT_DIRECTORY / 'v1.txt'
PROMPT_V2_PATH = PROMPT_DIRECTORY / 'v2.txt'

required_prompt_files = [
    PROMPT_V1_PATH, PROMPT_V2_PATH
]
missing_prompt_files = [
    str(path) for path in required_prompt_files if not path.exists()
]

if missing_prompt_files:
    raise FileNotFoundError(
        'Missing versioned prompt files: ' + ', '.join(missing_prompt_files)
    )

PROMPT_V1 = PROMPT_V1_PATH.read_text(encoding='utf-8')
PROMPT_V2 = PROMPT_V2_PATH.read_text(encoding='utf-8')

print('Loaded prompt v1 from:', PROMPT_V1_PATH)
print('Loaded prompt v2 from:', PROMPT_V2_PATH)
print('v1 characters:', len(PROMPT_V1))
print('v2 characters:', len(PROMPT_V2))

Loaded prompt v1 from: /content/drive/MyDrive/prompt_versions/v1.txt
Loaded prompt v2 from: /content/drive/MyDrive/prompt_versions/v2.txt
v1 characters: 1720
v2 characters: 1825


# Part 2: Build the test suite



The test suite contains 12 findings that cover command injection, password resets, broken authorization, information exposure, authentication bypass, stored XSS, account enumeration, network controls, SQL injection, CSRF defense, sensitive-data disclosure, and resource exhaustion. Each case has an expected severity and rationale.

In [6]:
import pandas as pd

TEST_CASES = [
    {'id':'T01','input':'A public image upload concatenates the filename into a root shell command without escaping.','expected_severity':'CRITICAL','expected_rationale':'Unauthenticated command injection permits arbitrary code execution with root privileges.'},
    {'id':'T02','input':'A reset token is random, single-use, and expires after 24 hours instead of 30 minutes.','expected_severity':'MEDIUM','expected_rationale':'The excessive token lifetime increases misuse opportunity, although an attacker must first obtain the token.'},
    {'id':'T03','input':'Any logged-in customer can edit another customer address by changing user_id.','expected_severity':'HIGH','expected_rationale':'Broken object authorization lets authenticated users modify other users account data.'},
    {'id':'T04','input':'A VPN-only admin page shows the application version and build date.','expected_severity':'LOW','expected_rationale':'Only trusted administrators can see low-sensitivity version information.'},
    {'id':'T05','input':'The production /admin/export endpoint trusts X-Admin: true and otherwise has no authentication; it downloads all customer tax records.','expected_severity':'CRITICAL','expected_rationale':'A trivial authentication bypass on a production admin endpoint exposes all customers highly sensitive tax records.'},
    {'id':'T06','input':'User markdown is rendered without sanitization, but scripts execute only after an administrator opens the report.','expected_severity':'HIGH','expected_rationale':'Stored cross-site scripting can compromise an administrator, though it requires administrator interaction.'},
    {'id':'T07','input':'Login responses reveal whether an email is registered, with rate limiting of five attempts per hour.','expected_severity':'LOW','expected_rationale':'Account enumeration has limited impact and strong rate limiting reduces practical exploitation.'},
    {'id':'T08','input':'A route named /admin/health has no application login, but ingress restricts it to localhost; it returns only OK.','expected_severity':'LOW','expected_rationale':'Network controls restrict the endpoint to localhost and the response contains no sensitive information.'},
    {'id':'T09','input':'A SQL query concatenates a report name, but only finance managers have access and the database role is read-only.','expected_severity':'HIGH','expected_rationale':'SQL injection can expose broad financial data despite privileged access and a read-only role.'},
    {'id':'T10','input':'Session cookies omit SameSite, while CSRF tokens protect every state-changing request.','expected_severity':'LOW','expected_rationale':'The missing cookie attribute is defense in depth because CSRF tokens protect state changes.'},
    {'id':'T11','input':'An unauthenticated GraphQL query returns every user name, email, password hash, and recovery codes.','expected_severity':'CRITICAL','expected_rationale':'A public endpoint broadly discloses password hashes and account recovery secrets.'},
    {'id':'T12','input':'An authenticated user can trigger PDF generation repeatedly; quotas allow 100 jobs per minute.','expected_severity':'MEDIUM','expected_rationale':'Weak quotas allow resource exhaustion, but authentication and per-user limits constrain the attack.'}
]

tests_df = pd.DataFrame(TEST_CASES)
print('Test cases:', len(tests_df))

Test cases: 12


# fixtures

These labeled outputs are used only when LIVE is False. They make the offline experiment deterministic: v1 under-classifies T05; v2 corrects T05 but overgeneralizes its special rule and regresses on T08.



In [7]:
FIXTURES = {
    'v1': {
        'T01':('CRITICAL','Public command injection enables arbitrary code execution as root.'),
        'T02':('MEDIUM','A 24-hour reset window increases misuse risk, although the token must be acquired and is single-use.'),
        'T03':('HIGH','Any customer can modify another customer data through broken object authorization.'),
        'T04':('LOW','The VPN restriction limits low-sensitivity build information to administrators.'),
        'T05':('HIGH','The spoofable header exposes a sensitive customer export without real authentication.'),
        'T06':('HIGH','Stored script execution can compromise an administrator after the report is opened.'),
        'T07':('LOW','The leak permits account enumeration, but strict rate limiting limits practical abuse.'),
        'T08':('LOW','Localhost ingress restriction and a non-sensitive OK response make impact minimal.'),
        'T09':('HIGH','A finance manager can exploit SQL injection to read broad financial information.'),
        'T10':('LOW','CSRF tokens mitigate state-changing requests, leaving SameSite as defense in depth.'),
        'T11':('CRITICAL','An unauthenticated query exposes password hashes and recovery codes for all users.'),
        'T12':('MEDIUM','High authenticated quotas permit resource pressure, but access and limits constrain it.')
    },
    'v2': {
        'T01':('CRITICAL','Public command injection enables arbitrary code execution as root.'),
        'T02':('MEDIUM','A 24-hour reset window increases misuse risk, although the token must be acquired and is single-use.'),
        'T03':('HIGH','Any customer can modify another customer data through broken object authorization.'),
        'T04':('LOW','The VPN restriction limits low-sensitivity build information to administrators.'),
        'T05':('CRITICAL','A trivial authentication bypass exposes all customer tax records through a production admin endpoint.'),
        'T06':('HIGH','Stored script execution can compromise an administrator after the report is opened.'),
        'T07':('LOW','The leak permits account enumeration, but strict rate limiting limits practical abuse.'),
        'T08':('CRITICAL','The admin endpoint lacks application authentication, so the special rule makes it critical despite localhost controls.'),
        'T09':('HIGH','A finance manager can exploit SQL injection to read broad financial information.'),
        'T10':('LOW','CSRF tokens mitigate state-changing requests, leaving SameSite as defense in depth.'),
        'T11':('CRITICAL','An unauthenticated query exposes password hashes and recovery codes for all users.'),
        'T12':('MEDIUM','High authenticated quotas permit resource pressure, but access and limits constrain it.')
    }
}



run_case() sends a structured system/user message to the live model or retrieves the appropriate fixture. Live responses are parsed as JSON. A small regular-expression fallback handles a model that returns the earlier SEVERITY: / RATIONALE: text format. Invalid output is labeled PARSE_ERROR instead of silently passing.

In [8]:
import re
def parse_model_response(text):
    try:
        data = json.loads(text)
        return data.get('severity', ''), data.get('rationale', '')
    except (json.JSONDecodeError, TypeError):
        severity = re.search(r'^SEVERITY:\s*(\w+)', text or '', re.MULTILINE)
        rationale = re.search(r'^RATIONALE:\s*(.+)', text or '', re.MULTILINE)
        if severity and rationale:
            return severity.group(1), rationale.group(1)
        return 'PARSE_ERROR', text or ''


def run_case(version, prompt, case):
    if LIVE:
        response_text = gemini_chat([
            {'role': 'system', 'content': prompt},
            {'role': 'user', 'content': 'Finding: ' + case['input']}
        ])
        return parse_model_response(response_text)
    return FIXTURES[version][case['id']]

A valid case returns exactly two values: a severity label and a rationale. A PARSE_ERROR result indicates format collapse and should count as an exact-match failure.

evaluate_prompt() runs all 12 cases, computes exact-match and semantic-similarity scores, and records the actual and expected values in a DataFrame. The cell evaluates both versions and previews the first five v1 results.

In [9]:
def evaluate_prompt(version, prompt):
    rows = []
    for case in TEST_CASES:
        predicted_severity, actual_rationale = run_case(version, prompt, case)
        rows.append({
            'version': version,
            'id': case['id'],
            'expected_severity': case['expected_severity'],
            'predicted_severity': predicted_severity,
            'exact_match': exact_match(case['expected_severity'], predicted_severity),
            'semantic_similarity': semantic_sim(case['expected_rationale'], actual_rationale),
            'expected_rationale': case['expected_rationale'],
            'actual_rationale': actual_rationale
        })
    return pd.DataFrame(rows)


v1_results = evaluate_prompt('v1', PROMPT_V1)
v2_results = evaluate_prompt('v2', PROMPT_V2)

# Part 3: Evaluate and show the trade-off

This cell combines the results and calculates each prompt version's overall exact-match accuracy and mean semantic similarity.

In [10]:
all_results = pd.concat([v1_results, v2_results], ignore_index=True)
summary = all_results.groupby('version').agg(
    exact_match_accuracy=('exact_match', 'mean'),
    mean_semantic_similarity=('semantic_similarity', 'mean')
).round(4)
summary

,exact_match_accuracy,mean_semantic_similarity
version,,
v1,0.9167,0.6576
v2,0.9167,0.6962


Result Explanation

Both Prompt v1 and prompt v2 achieved 91.67% exact-match accuracy. Across 12 cases, these percentages correspond to 11 correct severity labels for both v1 and v2.

# Locate improved and regressed cases

This cell merges v1 and v2 by test ID and automatically labels cases whose exact-match result changed as IMPROVED or REGRESSED.



In [11]:
comparison = v1_results.merge(v2_results, on='id', suffixes=('_v1', '_v2'))
changed = comparison.loc[
    comparison['exact_match_v1'] != comparison['exact_match_v2']
].copy()
changed['outcome'] = changed.apply(
    lambda row: 'IMPROVED' if row['exact_match_v2'] > row['exact_match_v1'] else 'REGRESSED',
    axis=1
)

tradeoff = changed[[
    'id', 'expected_severity_v1', 'predicted_severity_v1',
    'exact_match_v1', 'semantic_similarity_v1','expected_severity_v2',
    'predicted_severity_v2', 'exact_match_v2',
    'semantic_similarity_v2', 'outcome'
]]
tradeoff

,id,expected_severity_v1,predicted_severity_v1,exact_match_v1,semantic_similarity_v1,expected_severity_v2,predicted_severity_v2,exact_match_v2,semantic_similarity_v2,outcome
4,T05,CRITICAL,HIGH,0.0,0.524,CRITICAL,CRITICAL,1.0,0.953,IMPROVED
7,T08,LOW,LOW,1.0,0.614,LOW,CRITICAL,0.0,0.649,REGRESSED


Result Explanation:

T05 improved: v1 predicted HIGH, while v2 predicts CRITICAL. Exact match changes from 0 to 1.

T08 regressed: v1 predicted the expected LOW, while v2 predicts CRITICAL. Exact match changes from 1 to 0.



# Part 4: Explain the failure and propose a resolution

Failure:

The v2 edit helped T05 because it forced to corrected v1's under-classification, However hurted T08. Therefore, the edit improved one of the classification accuracy, and regressed the other one, while making the generated rationales less similar to the reference rationales on these two cases. This directly demonstrates that the prompt edit created a tradeoff rather than producing a universal improvement.

 Proposed resolution:

T05 and T08 should then be added as contrastive few-shot examples. Together, they show the decision boundary: genuine public authentication bypass with sensitive impact versus a harmless, network-restricted health endpoint.


# Part 5: Submit

Store the prompt versions as files, run the suite (set your key for real calls), and open a pull request with the metric numbers and a linked research note. Rubric: versioned prompts (15), structured prompt (20), test suite with two metrics (25), tradeoff with numbers (25), PR hygiene (15).
